# Runbook 4.2.b: Multi-factor authentication  (MODEL SOLUTION, instructor copy, do not distribute)

**Skill:** Mitigate Risk · **Maps to:** 4.2.A, 4.2.B · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** A security analyst receives, for each login attempt, the list of authentication factors that were
presented (e.g. `["password", "sms_code"]`). Build the classifier that determines whether the
attempt satisfies true multi-factor authentication, and if so, whether it relies on a
phishing-resistant possession factor or a phishable one.

Categories of result:
- `"single_factor"`: the presented factors, once mapped to their factor category (knowledge /
  possession / inherence), cover fewer than two distinct categories. This includes presenting only
  one factor, presenting several factors from the *same* category (e.g. two passwords), an empty
  list, or a list of only unrecognized factor strings.
- `"mfa_phishing_resistant"`: two or more distinct factor categories are present, and at least one
  of the presented factors is a phishing-resistant possession factor (`hardware_key` or `fido2`).
- `"mfa_phishable"`: two or more distinct factor categories are present, but no phishing-resistant
  possession factor was used (e.g. it relies on `sms_code`, `totp_code`, `authenticator_app`,
  `push_approved`, or `backup_code` for the possession factor).

The exact factor-to-category mapping and full rule set are defined in the Reference section below, implementations must follow it precisely.

**Lesson objective.** Explain why single-factor (password-only) login is insufficient, classify a described authentication method into its factor category, judge whether a login attempt satisfies true multi-factor authentication, and identify which MFA methods are phishing-resistant versus phishable.

### What comes in
A list of strings, each a recognized or unrecognized factor name. May be empty.

### What must come out
One of: `"single_factor"`, `"mfa_phishable"`, `"mfa_phishing_resistant"`.

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```
["password"]                          -> "single_factor"
["password", "password"]              -> "single_factor"
["password", "security_question"]     -> "single_factor"
["password", "sms_code"]              -> "mfa_phishable"
["password", "push_approved"]         -> "mfa_phishable"
["password", "hardware_key"]          -> "mfa_phishing_resistant"
["password", "sms_code", "hardware_key"] -> "mfa_phishing_resistant"
["biometric", "password"]             -> "mfa_phishable"
[]                                     -> "single_factor"
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only.
- [ ] Follow the exact factor -> category mapping in the Reference section below.
- [ ] Unrecognized factor strings must not raise an error; they are simply ignored.
- [ ] Must pass all cases in the tests (visible + hidden).
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(factors: list[str]) -> str
```

#### Input (`factors`)
A list of strings, each naming an authentication factor presented during one login attempt, e.g.
`["password", "hardware_key"]`. Order does not matter. The list may be empty.

#### Factor -> category mapping (exact, case-sensitive strings)
| factor string | category | phishing-resistant? |
|---|---|---|
| `password` | knowledge | n/a |
| `pin` | knowledge | n/a |
| `security_question` | knowledge | n/a |
| `sms_code` | possession | no |
| `totp_code` | possession | no |
| `authenticator_app` | possession | no |
| `push_approved` | possession | no (unverified push approval; MFA-fatigue risk) |
| `backup_code` | possession | no |
| `hardware_key` | possession | **yes** |
| `fido2` | possession | **yes** |
| `biometric` | inherence | no |

Any string not in this table is an **unrecognized factor** and is ignored when determining
categories (it does not count toward the category set and does not trigger phishing-resistant
status), but it does not cause an error.

#### Output (`result`)
Exactly one string from `{"single_factor", "mfa_phishable", "mfa_phishing_resistant"}`.

#### Classification rules
1. Map every string in `factors` to a category using the table above; unrecognized strings map to
   no category and are skipped.
2. Let `distinct_categories` be the set of distinct categories (`knowledge`, `possession`,
   `inherence`) actually present among recognized factors.
3. If `len(distinct_categories) < 2` (this includes an empty list, an all-unrecognized list, and a
   list where every recognized factor maps to the same single category, e.g. two knowledge
   credentials): return `"single_factor"`.
4. Otherwise (2 or more distinct categories are present):
   - If any recognized factor is a phishing-resistant possession factor (`hardware_key` or
     `fido2`): return `"mfa_phishing_resistant"`.
   - Otherwise: return `"mfa_phishable"`.

Phishing-resistance takes priority: if a phishing-resistant possession factor is present alongside
any number of phishable factors, the result is still `"mfa_phishing_resistant"`.

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Does not raise on empty lists or unrecognized factor strings.
- Returns a single string (not a list).

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

This is the model solution.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
from typing import List, Optional

_CATEGORY = {
    "password": "knowledge",
    "pin": "knowledge",
    "security_question": "knowledge",
    "sms_code": "possession",
    "totp_code": "possession",
    "authenticator_app": "possession",
    "push_approved": "possession",
    "backup_code": "possession",
    "hardware_key": "possession",
    "fido2": "possession",
    "biometric": "inherence",
}

_PHISHING_RESISTANT = {"hardware_key", "fido2"}


def _category_of(factor: str) -> Optional[str]:
    return _CATEGORY.get(factor)


def solve(factors: List[str]) -> str:
    if not factors:
        return "single_factor"

    distinct_categories = set()
    has_phishing_resistant = False

    for factor in factors:
        category = _category_of(factor)
        if category is None:
            continue
        distinct_categories.add(category)
        if factor in _PHISHING_RESISTANT:
            has_phishing_resistant = True

    if len(distinct_categories) < 2:
        return "single_factor"

    if has_phishing_resistant:
        return "mfa_phishing_resistant"

    return "mfa_phishable"


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

def test_example_password_only():
    assert impl.solve(["password"]) == "single_factor"


def test_example_two_passwords():
    assert impl.solve(["password", "password"]) == "single_factor"


def test_example_password_security_question():
    assert impl.solve(["password", "security_question"]) == "single_factor"


def test_example_password_sms():
    assert impl.solve(["password", "sms_code"]) == "mfa_phishable"


def test_example_password_push():
    assert impl.solve(["password", "push_approved"]) == "mfa_phishable"


def test_example_password_hardware_key():
    assert impl.solve(["password", "hardware_key"]) == "mfa_phishing_resistant"


def test_edge_empty_list():
    assert impl.solve([]) == "single_factor"


def test_edge_unknown_factor_only():
    assert impl.solve(["carrier_pigeon"]) == "single_factor"


def test_edge_unknown_plus_known_single_category():
    assert impl.solve(["password", "carrier_pigeon"]) == "single_factor"


def test_biometric_plus_password_is_phishable_not_resistant():
    assert impl.solve(["biometric", "password"]) == "mfa_phishable"


def test_fido2_string_variant():
    assert impl.solve(["password", "fido2"]) == "mfa_phishing_resistant"


def test_priority_hardware_key_wins_over_phishable_factor():
    assert impl.solve(["password", "sms_code", "hardware_key"]) == "mfa_phishing_resistant"


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Run your finished tool on the examples from the brief, then change one value and predict the answer before you run it again.

- [ ] Every example says OK
- [ ] Your own changed example gives the answer you predicted

In [ ]:
checks = [
    (lambda: solve(["password"]),
     "single_factor"),
    (lambda: solve(["password", "password"]),
     "single_factor"),
    (lambda: solve(["password", "security_question"]),
     "single_factor"),
    (lambda: solve(["password", "sms_code"]),
     "mfa_phishable"),
    (lambda: solve(["password", "push_approved"]),
     "mfa_phishable"),
    (lambda: solve(["password", "hardware_key"]),
     "mfa_phishing_resistant"),
    (lambda: solve(["password", "sms_code", "hardware_key"]),
     "mfa_phishing_resistant"),
    (lambda: solve(["biometric", "password"]),
     "mfa_phishable"),
    (lambda: solve([]),
     "single_factor"),
]

for n, (run, want) in enumerate(checks, 1):
    try:
        got = run()
    except Exception as e:
        got = "error: " + repr(e)
    print("example", n, "OK  " if got == want else "DIFF", "got:", got, " expected:", want)


## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it